<a href="https://colab.research.google.com/github/Almogod/Text-to-image-GAN/blob/main/GAN.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

!pip install torch torchaudio torchvision --index-url https://download.pytorch.org/whl/cu118
!pip install diffusers==0.21.0 transformers==4.30.2 accelerate==0.20.3
!pip install safetensors==0.3.1 xformers==0.0.20 Pillow==9.5.0
!pip install numpy==1.24.4  matplotlib==3.7.2  gradio==4.0.0

Looking in indexes: https://download.pytorch.org/whl/cu118
  Using cached diffusers-0.21.0-py3-none-any.whl
  Using cached transformers-4.30.2-py3-none-any.whl.metadata (113 kB)
  Using cached accelerate-0.20.3-py3-none-any.whl.metadata (17 kB)
  Using cached huggingface_hub-0.36.2-py3-none-any.whl.metadata (15 kB)
  Using cached tokenizers-0.13.3.tar.gz (314 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
Using cached transformers-4.30.2-py3-none-any.whl (7.2 MB)
Using cached accelerate-0.20.3-py3-none-any.whl (227 kB)
Using cached huggingface_hub-0.36.2-py3-none-any.whl (566 kB)
  error: subprocess-exited-with-error
  
  × Building wheel for tokenizers (pyproject.toml) did not run successfully.
  │ exit code: 1
  ╰─> See above for output.
  
  note: This error originates from a subprocess, and is likely not a problem with pip.
  ERROR: Failed building wheel for tokenizers
Failed to build tokeni

In [ ]:
import torch
import torch.nn.functional as f
from diffusers import (
    StableDiffusionPipeline,
    EulerAncestralDiscreteScheduler,
    LMSDiscreteScheduler,
    DPMSolverMultistepScheduler,
    EulerDiscreteScheduler,
    DDIMScheduler
)
import gradio as gr
import numpy as np
from torch import autocast
import os, time
from typing import List, Tuple, Optional
from datetime import datetime
from PIL import Image

Now that the dependencies are installed and modules imported, let's load a pre-trained Stable Diffusion model. We'll use `runwayml/stable-diffusion-v1-5` and set up `EulerAncestralDiscreteScheduler` for image generation. We'll also configure it to use GPU if available.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load the Stable Diffusion pipeline
pipeline = StableDiffusionPipeline.from_pretrained("runwayml/stable-diffusion-v1-5", torch_dtype=torch.float16)
pipeline.scheduler = EulerAncestralDiscreteScheduler.from_config(pipeline.scheduler.config)
pipeline.to(device)

Using device: cuda


model_index.json:   0%|          | 0.00/541 [00:00<?, ?B/s]

Fetching 15 files:   0%|          | 0/15 [00:00<?, ?it/s]

Loading pipeline components...:   0%|          | 0/7 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/396 [00:00<?, ?it/s]

StableDiffusionSafetyChecker LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/safety_checker
Key                                               | Status     |  | 
--------------------------------------------------+------------+--+-
vision_model.vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loading weights:   0%|          | 0/196 [00:00<?, ?it/s]

CLIPTextModel LOAD REPORT from: /root/.cache/huggingface/hub/models--runwayml--stable-diffusion-v1-5/snapshots/451f4fe16113bff5a5d2269ed5ad43b0592e9a14/text_encoder
Key                                | Status     |  | 
-----------------------------------+------------+--+-
text_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


StableDiffusionPipeline {
  "_class_name": "StableDiffusionPipeline",
  "_diffusers_version": "0.37.0",
  "_name_or_path": "runwayml/stable-diffusion-v1-5",
  "feature_extractor": [
    "transformers",
    "CLIPImageProcessor"
  ],
  "image_encoder": [
    null,
    null
  ],
  "requires_safety_checker": true,
  "safety_checker": [
    "stable_diffusion",
    "StableDiffusionSafetyChecker"
  ],
  "scheduler": [
    "diffusers",
    "EulerAncestralDiscreteScheduler"
  ],
  "text_encoder": [
    "transformers",
    "CLIPTextModel"
  ],
  "tokenizer": [
    "transformers",
    "CLIPTokenizer"
  ],
  "unet": [
    "diffusers",
    "UNet2DConditionModel"
  ],
  "vae": [
    "diffusers",
    "AutoencoderKL"
  ]
}

Next, let's define the core image generation function. This function will take a text prompt, optional negative prompt, and other parameters to generate an image using the loaded Stable Diffusion model. It implicitly handles text preprocessing and embedding creation.

In [ ]:
def generate_image(
    prompt: str,
    negative_prompt: str = "",
    num_inference_steps: int = 50,
    guidance_scale: float = 7.5,
    seed: int = -1
) -> Image.Image:
    if seed == -1:
        seed = np.random.randint(0, 1000000)
    generator = torch.Generator(device=device).manual_seed(seed)

    with autocast(device):
        image = pipeline(
            prompt,
            negative_prompt=negative_prompt,
            num_inference_steps=num_inference_steps,
            guidance_scale=guidance_scale,
            generator=generator
        ).images[0]
    return image

Finally, we will create a Gradio interface to allow easy interaction with our text-to-image model. You can input your prompt, an optional negative prompt, adjust inference steps, guidance scale, and set a random seed.

In [ ]:
iface = gr.Interface(
    fn=generate_image,
    inputs=[
        gr.Textbox(label="Prompt", placeholder="A vibrant watercolor painting of a futuristic city at sunset"),
        gr.Textbox(label="Negative Prompt (optional)", placeholder="blurry, low quality, bad anatomy"),
        gr.Slider(minimum=10, maximum=100, step=1, value=50, label="Inference Steps"),
        gr.Slider(minimum=1.0, maximum=20.0, step=0.5, value=7.5, label="Guidance Scale"),
        gr.Number(label="Seed (-1 for random)", value=-1)
    ],
    outputs="image",
    title="Text-to-Image with Stable Diffusion",
    description="Generate images from text prompts using Stable Diffusion 1.5. Adjust parameters to fine-tune your results."
)

iface.launch(debug=True)

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://2d76532bca716d8814.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

  0%|          | 0/50 [00:00<?, ?it/s]

Keyboard interruption in main thread... closing server.
Killing tunnel 127.0.0.1:7860 <> https://2d76532bca716d8814.gradio.live
